# Step 8 — Bird's-Eye-View Fusion Video

| | |
|---|---|
| **Input** | `output/step_0` (samples index, ego pose), `output/step_1/lidar` + `output/step_2/lidar` (LiDAR clusters), `output/step_2/radar` (post-declutter radar points), `output/step_2/yolo_mono` (monocular camera detections), `output/step_1/camera_meta.json` + raw `CAM_FRONT` JPEGs (`DATA_ROOT`), `output/step_4/fused_tracks_all.csv`, `output/step_5/ttc_fused.csv` |
| **Outputs** | `output/step_8/videos/<scene_name>.mp4` — one video per nuScenes scene (10 total) |
| | `output/step_8/bev_video_summary.csv` — per-scene frame count / fps / filename |
| **Used by** | Nothing downstream — this is a terminal visualization stage |

## What this notebook does

Renders one combined frame per sample: a **Bird's-Eye View** (BEV) panel on the
left showing every sensor's detections for that frame — LiDAR cluster centroids,
post-declutter radar points, and true-monocular (`camera_mono`) camera
detections — all transformed into the ego vehicle's own frame (ego at the
origin, heading pointing "up"), plus the **fused** tracks active that frame
with their fused TTC value labeled and color-coded by urgency (red < 3s,
orange < 6s, yellow < 10s, green otherwise, gray = no finite TTC yet). The
right panel shows the real `CAM_FRONT` image for that sample — the actual
input video feed, for a direct sanity check against the BEV reconstruction.

Frames are then encoded into one MP4 per scene (nuScenes v1.0-mini's 404
samples are 10 separate continuous driving clips, not one single recording —
see `CLAUDE.md`'s "Video scope" discussion), at a per-scene fps derived from
that scene's own real sample-to-sample elapsed time (`timestamp_us`), so each
video reproduces the original recording's actual temporal cadence rather than
an assumed constant.

This stage is purely for visual inspection/presentation — it reads only
already-committed pipeline outputs and writes nothing any other stage
consumes.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Imports and output directories
# ─────────────────────────────────────────────────────────────────
import json
import shutil
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from config import DATA_ROOT, STEP0_DIR, STEP1_DIR, STEP2_DIR, STEP4_DIR, STEP5_DIR, STEP8_DIR
from src.geometry import transform_matrix

FRAMES_DIR = STEP8_DIR / "frames"
VIDEOS_DIR = STEP8_DIR / "videos"

# FIXED (same idempotency convention as Steps 3.1/3.2/3.3/4): clear this
# stage's own output dir before regenerating, so a re-run never mixes an old
# video/frame set with a new one.
if STEP8_DIR.exists():
    shutil.rmtree(STEP8_DIR)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

print(f"\u2705 Step 8 output dirs ready: {STEP8_DIR}")

config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output
✅ Step 8 output dirs ready: F:\Sensor fusion Research\output\step_8


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Named constants
# ─────────────────────────────────────────────────────────────────

BEV_RANGE_M = 60.0          # ego-centered BEV plot extent, metres each side
LIDAR_POINT_SIZE = 4
RADAR_MARKER_SIZE = 60
CAMERA_MARKER_SIZE = 50
FUSED_MARKER_SIZE = 140

# TTC urgency color bands (seconds) -- purely a visual color-coding choice
# for this BEV, not a scoring threshold (Step 6's Good_pct uses its own).
TTC_RED_S = 3.0
TTC_ORANGE_S = 6.0
TTC_YELLOW_S = 10.0

CAM_PANEL_CHANNEL = "CAM_FRONT"   # the "video as it was in input" reference panel
CAMERA_CHANNELS = ["CAM_FRONT", "CAM_FRONT_LEFT", "CAM_FRONT_RIGHT",
                    "CAM_BACK", "CAM_BACK_LEFT", "CAM_BACK_RIGHT"]
RADAR_CHANNELS = ["RADAR_FRONT", "RADAR_FRONT_LEFT", "RADAR_FRONT_RIGHT",
                   "RADAR_BACK_LEFT", "RADAR_BACK_RIGHT"]

FPS_MIN, FPS_MAX = 1.0, 10.0   # clamp a scene's measured fps to a sane range

print(f"\u2705 BEV_RANGE_M={BEV_RANGE_M}m, TTC color bands: red<{TTC_RED_S}s, orange<{TTC_ORANGE_S}s, yellow<{TTC_YELLOW_S}s")

✅ BEV_RANGE_M=60.0m, TTC color bands: red<3.0s, orange<6.0s, yellow<10.0s


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Per-sample data loaders
#
# Every loader below returns points already in the GLOBAL frame (metres),
# using the same point_to_global-style transform chain already established
# in src/geometry.py and used throughout Steps 1-4 -- nothing new here,
# just reused for visualization instead of tracking.
# ─────────────────────────────────────────────────────────────────

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

with open(STEP1_DIR / "camera_meta.json") as f:
    camera_meta = json.load(f)

fused_tracks_df = pd.read_csv(STEP4_DIR / "fused_tracks_all.csv")
ttc_fused_df = pd.read_csv(STEP5_DIR / "ttc_fused.csv")[["fused_id", "sample_id", "ttc"]]
# Left join: a fused point with no TTC yet (e.g. a track's very first frame,
# before enough history exists to estimate velocity) keeps ttc=NaN rather
# than being dropped -- rendered as "no TTC yet" (gray), not silently hidden.
fused_with_ttc = fused_tracks_df.merge(ttc_fused_df, on=["fused_id", "sample_id"], how="left")


def _apply_transform(M, points_xyz):
    """points_xyz: (N,3) -> (N,3), via a precomputed 4x4 homogeneous matrix."""
    pts = np.asarray(points_xyz, dtype=float)
    if len(pts) == 0:
        return pts
    pts_h = np.hstack([pts, np.ones((len(pts), 1))]).T
    return (M @ pts_h)[:3].T


def load_ego_pose(sample_id):
    with open(STEP0_DIR / sample_id / "ego_pose.json") as f:
        return json.load(f)


def load_lidar_points_global(sample_id):
    """Non-ground LiDAR cluster centroids, lifted to the global frame."""
    clusters_path = STEP2_DIR / "lidar" / sample_id / "lidar_clusters.json"
    meta_path = STEP1_DIR / "lidar" / sample_id / "lidar_meta.json"
    if not clusters_path.exists() or not meta_path.exists():
        return np.empty((0, 3))
    with open(clusters_path) as f:
        clusters = json.load(f)
    if not clusters:
        return np.empty((0, 3))
    with open(meta_path) as f:
        lidar_meta = json.load(f)
    ego_pose = lidar_meta["ego_pose"]
    calib = {"translation": lidar_meta["sensor_to_ego_translation"],
              "rotation": lidar_meta["sensor_to_ego_rotation"]}
    T = transform_matrix(ego_pose["translation"], ego_pose["rotation"]) @ \
        transform_matrix(calib["translation"], calib["rotation"])
    centroids = np.array([v["centroid"] for v in clusters.values()])
    return _apply_transform(T, centroids)


def load_radar_points_global(sample_id):
    """All kept (post-declutter) radar points across all 5 channels, global frame."""
    out = []
    radar_dir = STEP2_DIR / "radar" / sample_id
    if not radar_dir.exists():
        return np.empty((0, 3))
    for chan in RADAR_CHANNELS:
        p = radar_dir / f"{chan}.json"
        if not p.exists():
            continue
        with open(p) as f:
            d = json.load(f)
        pts = d.get("points", [])
        if not pts:
            continue
        calib = d["calibration"]
        T = transform_matrix(calib["ego_pose"]["translation"], calib["ego_pose"]["rotation"]) @ \
            transform_matrix(calib["sensor_to_ego_translation"], calib["sensor_to_ego_rotation"])
        local = np.array([[pt["x"], pt["y"], pt["z"]] for pt in pts])
        out.append(_apply_transform(T, local))
    if not out:
        return np.empty((0, 3))
    return np.vstack(out)


def load_camera_mono_points_global(sample_id):
    """All 6 cameras' true-monocular (camera_mono) detections, already global."""
    out = []
    cam_dir = STEP2_DIR / "yolo_mono" / sample_id
    if not cam_dir.exists():
        return np.empty((0, 3))
    for chan in CAMERA_CHANNELS:
        p = cam_dir / f"{chan}.json"
        if not p.exists():
            continue
        with open(p) as f:
            dets = json.load(f)
        for det in dets:
            if det.get("has_3d_position"):
                out.append([det["global_x"], det["global_y"], det["global_z"]])
    if not out:
        return np.empty((0, 3))
    return np.array(out)


def load_fused_tracks_for_sample(sample_id):
    """Fused (x, y, ttc) for every fused track active in this sample."""
    rows = fused_with_ttc[fused_with_ttc["sample_id"] == sample_id]
    return rows[["fused_id", "x", "y", "ttc"]].to_dict("records")


def load_cam_front_image(sample_id):
    meta = camera_meta.get(sample_id, {}).get(CAM_PANEL_CHANNEL)
    if meta is None:
        return None
    path = DATA_ROOT / meta["filename"]
    if not path.exists():
        return None
    return Image.open(path)


print(f"\u2705 Loaded samples_index ({len(samples_index)}), camera_meta, fused_tracks+ttc ({len(fused_with_ttc)} rows)")

✅ Loaded samples_index (404), camera_meta, fused_tracks+ttc (40297 rows)


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Rendering: BEV panel, camera panel, combined frame
# ─────────────────────────────────────────────────────────────────

def ttc_to_color(ttc):
    if ttc is None or (isinstance(ttc, float) and (np.isnan(ttc) or np.isinf(ttc))):
        return "#888888"
    if ttc < TTC_RED_S:
        return "#d62728"
    if ttc < TTC_ORANGE_S:
        return "#ff7f0e"
    if ttc < TTC_YELLOW_S:
        return "#bcbd22"
    return "#2ca02c"


def to_ego_frame(points_global, ego_pose):
    """GLOBAL -> ego-local frame (ego at origin, x=forward, y=left), the
    same transform chain as src/geometry.py's point_to_global(), inverted."""
    if len(points_global) == 0:
        return np.empty((0, 3))
    T_inv = transform_matrix(ego_pose["translation"], ego_pose["rotation"], inverse=True)
    return _apply_transform(T_inv, points_global)


def render_bev(ax, sample_id, ego_pose):
    r = BEV_RANGE_M
    ax.set_xlim(-r, r)
    ax.set_ylim(-r, r)
    ax.set_aspect("equal")
    ax.set_facecolor("#0d1117")
    ax.grid(True, color="#30363d", linewidth=0.5)

    lidar_e = to_ego_frame(load_lidar_points_global(sample_id), ego_pose)
    if len(lidar_e):
        ax.scatter(-lidar_e[:, 1], lidar_e[:, 0], s=LIDAR_POINT_SIZE, c="#8b949e",
                   alpha=0.6, label="LiDAR", zorder=2)

    radar_e = to_ego_frame(load_radar_points_global(sample_id), ego_pose)
    if len(radar_e):
        ax.scatter(-radar_e[:, 1], radar_e[:, 0], s=RADAR_MARKER_SIZE, c="#ff7f0e",
                   marker="^", edgecolors="none", alpha=0.85, label="Radar", zorder=3)

    cam_e = to_ego_frame(load_camera_mono_points_global(sample_id), ego_pose)
    if len(cam_e):
        ax.scatter(-cam_e[:, 1], cam_e[:, 0], s=CAMERA_MARKER_SIZE, c="#58a6ff",
                   marker="s", edgecolors="none", alpha=0.85, label="Camera (mono)", zorder=3)

    for row in load_fused_tracks_for_sample(sample_id):
        pos_e = to_ego_frame(np.array([[row["x"], row["y"], 0.0]]), ego_pose)[0]
        px, py = -pos_e[1], pos_e[0]
        if abs(px) > r or abs(py) > r:
            continue
        ttc = row["ttc"]
        color = ttc_to_color(ttc)
        ax.scatter([px], [py], s=FUSED_MARKER_SIZE, facecolors="none", edgecolors=color,
                   linewidths=2.2, zorder=4)
        label = f"{ttc:.1f}s" if ttc is not None and not (np.isnan(ttc) or np.isinf(ttc)) else "\u2014"
        ax.annotate(label, (px, py), textcoords="offset points", xytext=(6, 6),
                    fontsize=8, color=color, fontweight="bold")

    ego_tri = plt.Polygon([(0, 2.2), (-1.3, -1.3), (1.3, -1.3)], closed=True,
                           color="#f0f6fc", zorder=5)
    ax.add_patch(ego_tri)

    ax.set_xlabel("lateral (m)", color="#c9d1d9")
    ax.set_ylabel("forward (m)", color="#c9d1d9")
    ax.tick_params(colors="#c9d1d9")
    for spine in ax.spines.values():
        spine.set_color("#30363d")
    ax.set_title("Bird's-Eye View - all sensors + fused TTC", color="#f0f6fc", fontsize=11)
    ax.legend(loc="upper right", fontsize=7, facecolor="#161b22", labelcolor="#c9d1d9", framealpha=0.85)


def render_camera_panel(ax, sample_id):
    img = load_cam_front_image(sample_id)
    ax.set_facecolor("#0d1117")
    if img is not None:
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, f"{CAM_PANEL_CHANNEL} image not found", ha="center", va="center",
                 color="#c9d1d9", transform=ax.transAxes)
    ax.set_title(f"{CAM_PANEL_CHANNEL} - input video", color="#f0f6fc", fontsize=11)
    ax.axis("off")


def render_frame(sample_id, out_path):
    ego_pose = load_ego_pose(sample_id)
    fig, (ax_bev, ax_cam) = plt.subplots(1, 2, figsize=(14, 6.5), dpi=110, facecolor="#0d1117")
    render_bev(ax_bev, sample_id, ego_pose)
    render_camera_panel(ax_cam, sample_id)
    fig.suptitle(sample_id, color="#f0f6fc", fontsize=10)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(out_path, facecolor=fig.get_facecolor())
    plt.close(fig)


print("\u2705 Rendering functions ready")

✅ Rendering functions ready


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Main loop: render every sample, encode one MP4 per scene
# ─────────────────────────────────────────────────────────────────

scenes = OrderedDict()
for sid in sorted(samples_index.keys()):
    scenes.setdefault(samples_index[sid]["scene_name"], []).append(sid)

summary_rows = []

for scene_name, sample_ids in scenes.items():
    scene_frames_dir = FRAMES_DIR / scene_name
    scene_frames_dir.mkdir(parents=True, exist_ok=True)

    frame_paths = []
    for sid in tqdm(sample_ids, desc=f"Rendering {scene_name}"):
        out_path = scene_frames_dir / f"{sid}.png"
        render_frame(sid, out_path)
        frame_paths.append(out_path)

    # fps derived from this scene's own real elapsed time between samples
    # (timestamp_us) -- reproduces the original recording's actual temporal
    # cadence rather than assuming a fixed 2Hz.
    timestamps = [samples_index[sid]["timestamp_us"] for sid in sample_ids]
    dts = np.diff(timestamps) / 1e6
    dts = dts[dts > 0]
    fps = float(np.clip(1.0 / np.median(dts), FPS_MIN, FPS_MAX)) if len(dts) else 2.0

    first_frame = cv2.imread(str(frame_paths[0]))
    h, w = first_frame.shape[:2]
    video_path = VIDEOS_DIR / f"{scene_name}.mp4"
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for fp in frame_paths:
        writer.write(cv2.imread(str(fp)))
    writer.release()

    shutil.rmtree(scene_frames_dir)   # frames were only scratch for encoding

    summary_rows.append({"scene_name": scene_name, "n_samples": len(sample_ids),
                          "fps": round(fps, 2), "video": video_path.name})
    print(f"\u2705 {scene_name}: {len(sample_ids)} frames @ {fps:.2f} fps -> {video_path.name}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(STEP8_DIR / "bev_video_summary.csv", index=False)
print(f"\n\u2705 Step 8 complete: {len(summary_rows)} scene videos written to {VIDEOS_DIR}")
summary_df

Rendering scene-0061:   0%|          | 0/39 [00:00<?, ?it/s]

Rendering scene-0061:   3%|▎         | 1/39 [00:00<00:30,  1.24it/s]

Rendering scene-0061:   5%|▌         | 2/39 [00:01<00:29,  1.27it/s]

Rendering scene-0061:   8%|▊         | 3/39 [00:02<00:28,  1.24it/s]

Rendering scene-0061:  10%|█         | 4/39 [00:03<00:29,  1.19it/s]

...

Rendering scene-1100:  98%|█████████▊| 39/40 [00:31<00:00,  1.23it/s]

Rendering scene-1100: 100%|██████████| 40/40 [00:32<00:00,  1.27it/s]

Rendering scene-1100: 100%|██████████| 40/40 [00:32<00:00,  1.24it/s]

✅ scene-1100: 40 frames @ 2.00 fps -> scene-1100.mp4

✅ Step 8 complete: 10 scene videos written to F:\Sensor fusion Research\output\step_8\videos


,scene_name,n_samples,fps,video
0,scene-0061,39,2.0,scene-0061.mp4
1,scene-0103,40,2.0,scene-0103.mp4
2,scene-0553,41,2.0,scene-0553.mp4
3,scene-0655,41,2.0,scene-0655.mp4
4,scene-0757,41,2.0,scene-0757.mp4
5,scene-0796,40,2.0,scene-0796.mp4
6,scene-0916,41,2.0,scene-0916.mp4
7,scene-1077,41,2.0,scene-1077.mp4
8,scene-1094,40,2.0,scene-1094.mp4
9,scene-1100,40,2.0,scene-1100.mp4
